# LangSmith 追踪与可观测性
## LangSmith Tracing & Observability

本notebook演示LangSmith的核心功能:
- **Trace**: 完整的Agent执行追踪
- **Span**: 单个LLM/工具/检索器调用的追踪
- **Feedback收集**: 用户反馈和自动评估
- **Dataset创建**: 从追踪记录构建评估数据集
- **Experiment对比**: A/B实验比较

In [ ]:
# 安装 (如果尚未安装)
# !pip install langsmith langchain-core

In [ ]:
import os
import uuid
import time
import json
from typing import List, Dict, Optional, Any
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from collections import defaultdict

# 尝试导入LangSmith
try:
    from langsmith import Client, traceable
    from langsmith.schemas import Run, Feedback, DataType
    LANGSMITH_AVAILABLE = True
    print("LangSmith SDK 已加载")
except ImportError:
    LANGSMITH_AVAILABLE = False
    print("LangSmith SDK 未安装。将使用模拟实现演示概念。")
    print("安装: pip install langsmith")

## 1. LangSmith模拟实现

以下实现了LangSmith核心数据结构和追踪逻辑的模拟版本。

In [ ]:
class RunType(Enum):
    """运行类型"""
    LLM = "llm"
    CHAIN = "chain"
    TOOL = "tool"
    RETRIEVER = "retriever"
    EMBEDDING = "embedding"
    EVALUATOR = "evaluator"


@dataclass
class Span:
    """
    LangSmith Span - 代表单个操作
    
    可以是一个LLM调用、工具调用、检索器调用等。
    """
    span_id: str
    name: str
    run_type: RunType
    parent_run_id: Optional[str] = None
    
    # 输入输出
    inputs: Dict[str, Any] = field(default_factory=dict)
    outputs: Optional[Dict[str, Any]] = None
    error: Optional[str] = None
    
    # 时间
    start_time: Optional[float] = None
    end_time: Optional[float] = None
    
    # 元数据
    tags: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    # Token使用
    token_usage: Optional[Dict[str, int]] = None
    
    # 子Span
    child_spans: List['Span'] = field(default_factory=list)
    
    @property
    def duration_ms(self) -> float:
        """持续时间 (毫秒)"""
        if self.start_time and self.end_time:
            return (self.end_time - self.start_time) * 1000
        return 0
    
    @property
    def is_error(self) -> bool:
        """是否是错误"""
        return self.error is not None
    
    def add_child(self, child: 'Span') -> None:
        """添加子Span"""
        child.parent_run_id = self.span_id
        self.child_spans.append(child)
    
    def to_dict(self) -> Dict[str, Any]:
        """转换为字典"""
        return {
            'id': self.span_id,
            'name': self.name,
            'run_type': self.run_type.value,
            'parent_run_id': self.parent_run_id,
            'inputs': self.inputs,
            'outputs': self.outputs,
            'error': self.error,
            'start_time': datetime.fromtimestamp(self.start_time).isoformat() if self.start_time else None,
            'end_time': datetime.fromtimestamp(self.end_time).isoformat() if self.end_time else None,
            'duration_ms': round(self.duration_ms, 2),
            'tags': self.tags,
            'token_usage': self.token_usage,
            'children': [c.to_dict() for c in self.child_spans],
        }


@dataclass
class Trace:
    """
    LangSmith Trace - 代表完整的执行链
    
    包含所有相关的Span，从根到子。
    """
    trace_id: str
    name: str
    root_span: Span
    
    # Feedback
    feedbacks: List[Dict[str, Any]] = field(default_factory=list)
    
    # 元数据
    session_id: Optional[str] = None
    user_id: Optional[str] = None
    tags: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    @property
    def total_duration_ms(self) -> float:
        """总持续时间"""
        return self.root_span.duration_ms
    
    @property
    def total_tokens(self) -> int:
        """总Token消耗"""
        return self._count_tokens(self.root_span)
    
    def _count_tokens(self, span: Span) -> int:
        """递归计算Token总数"""
        total = 0
        if span.token_usage:
            total += sum(span.token_usage.values())
        for child in span.child_spans:
            total += self._count_tokens(child)
        return total
    
    def add_feedback(
        self,
        key: str,
        score: float,
        comment: str = "",
        source: str = "user",
    ) -> None:
        """添加反馈"""
        self.feedbacks.append({
            'trace_id': self.trace_id,
            'key': key,
            'score': score,
            'comment': comment,
            'source': source,
            'created_at': datetime.now().isoformat(),
        })
    
    def print_tree(self, indent: int = 0) -> None:
        """打印追踪树"""
        self._print_span(self.root_span, indent)
    
    def _print_span(self, span: Span, indent: int) -> None:
        """递归打印Span树"""
        prefix = "  " * indent
        if indent == 0:
            prefix = "📊 "
        else:
            prefix = "  " * indent + "├─ "
        
        error_mark = " ❌" if span.is_error else ""
        token_str = f" (tokens: {sum(span.token_usage.values())})" if span.token_usage else ""
        
        print(f"{prefix}[{span.run_type.value}] {span.name} "
              f"({span.duration_ms:.0f}ms{token_str}){error_mark}")
        
        for child in span.child_spans:
            self._print_span(child, indent + 1)


print("LangSmith模拟数据结构实现完成")
print("提供: Trace, Span, RunType, Feedback")

## 2. 追踪演示: 模拟一次完整的RAG Agent执行

In [ ]:
import uuid
import time
import random

def simulate_rag_agent_trace(query: str, user_id: str = "user_001") -> Trace:
    """
    模拟一次完整的RAG Agent执行并生成LangSmith Trace
    
    执行步骤:
    1. [LLM] 查询理解和重写
    2. [EMBEDDING] 查询向量化
    3. [RETRIEVER] 向量检索
    4. [RETRIEVER] 重排序
    5. [LLM] 答案生成
    6. [TOOL] 事实核查
    """
    trace_id = str(uuid.uuid4())
    root_span_id = str(uuid.uuid4())
    
    # Root span: Agent执行
    root_start = time.time()
    root_span = Span(
        span_id=root_span_id,
        name="RAG Agent Execution",
        run_type=RunType.CHAIN,
        inputs={"query": query, "user_id": user_id},
        tags=["rag", "agent", "production"],
        metadata={"version": "v2.1.0", "environment": "production"},
    )
    
    # Span 1: 查询重写 (LLM)
    start = time.time()
    time.sleep(random.uniform(0.1, 0.3))  # 模拟延迟
    query_rewrite_span = Span(
        span_id=str(uuid.uuid4()),
        name="Query Rewrite",
        run_type=RunType.LLM,
        inputs={"original_query": query},
        outputs={"rewritten_query": f"优化后的: {query}"},
        token_usage={"input": len(query) // 2, "output": len(query) // 3},
    )
    query_rewrite_span.start_time = start
    query_rewrite_span.end_time = time.time()
    root_span.add_child(query_rewrite_span)
    
    # Span 2: 嵌入 (EMBEDDING)
    start = time.time()
    time.sleep(random.uniform(0.05, 0.15))
    embedding_span = Span(
        span_id=str(uuid.uuid4()),
        name="Generate Query Embedding",
        run_type=RunType.EMBEDDING,
        inputs={"text": query},
        outputs={"embedding_dim": 1536, "model": "text-embedding-3-small"},
        token_usage={"input": len(query) // 2},
    )
    embedding_span.start_time = start
    embedding_span.end_time = time.time()
    root_span.add_child(embedding_span)
    
    # Span 3: 向量检索 (RETRIEVER)
    start = time.time()
    time.sleep(random.uniform(0.1, 0.5))
    retriever_span = Span(
        span_id=str(uuid.uuid4()),
        name="Vector Search",
        run_type=RunType.RETRIEVER,
        inputs={"index": "rag_knowledge_base", "top_k": 10},
        outputs={
            "num_results": 10,
            "top_scores": [0.92, 0.88, 0.85, 0.81, 0.79],
            "source_ids": ["doc_142", "doc_88", "doc_201", "doc_55", "doc_310"],
        },
        metadata={"index": "faiss", "dimension": 1536},
    )
    retriever_span.start_time = start
    retriever_span.end_time = time.time()
    root_span.add_child(retriever_span)
    
    # Span 4: 重排序 (RETRIEVER)
    start = time.time()
    time.sleep(random.uniform(0.1, 0.3))
    reranker_span = Span(
        span_id=str(uuid.uuid4()),
        name="Re-rank Results",
        run_type=RunType.RETRIEVER,
        inputs={"candidates": 10, "top_n": 3},
        outputs={
            "reranked_ids": ["doc_88", "doc_142", "doc_201"],
            "reranked_scores": [0.94, 0.91, 0.87],
        },
        metadata={"reranker_model": "bge-reranker-v2"},
    )
    reranker_span.start_time = start
    reranker_span.end_time = time.time()
    root_span.add_child(reranker_span)
    
    # Span 5: 答案生成 (LLM)
    start = time.time()
    time.sleep(random.uniform(0.5, 2.0))
    generation_span = Span(
        span_id=str(uuid.uuid4()),
        name="Generate Answer",
        run_type=RunType.LLM,
        inputs={
            "query": query,
            "contexts_count": 3,
            "prompt_template": "rag_v2",
        },
        outputs={
            "answer": "这是关于该问题的详细回答...",
            "model": "gpt-4o",
        },
        token_usage={"input": 1200, "output": 350},
        metadata={"model": "gpt-4o", "temperature": 0.0},
    )
    generation_span.start_time = start
    generation_span.end_time = time.time()
    root_span.add_child(generation_span)
    
    # Span 6: 事实核查 (TOOL) - 可选的验证步骤
    start = time.time()
    time.sleep(random.uniform(0.2, 0.5))
    fact_check_span = Span(
        span_id=str(uuid.uuid4()),
        name="Fact Verification",
        run_type=RunType.TOOL,
        inputs={"answer": "...", "contexts": ["..."]},
        outputs={"verified_claims": 4, "total_claims": 5, "hallucination_detected": False},
        metadata={"verification_method": "claim_decomposition"},
    )
    fact_check_span.start_time = start
    fact_check_span.end_time = time.time()
    root_span.add_child(fact_check_span)
    
    # 完成Root Span
    root_span.start_time = root_start
    root_span.end_time = time.time()
    root_span.outputs = {
        "status": "success",
        "total_spans": 6,
        "answer_length": 350,
    }
    
    # 创建Trace
    trace = Trace(
        trace_id=trace_id,
        name=f"RAG: {query[:30]}...",
        root_span=root_span,
        session_id=str(uuid.uuid4()),
        user_id=user_id,
        tags=["production", "v2.1.0"],
        metadata={"source": "api", "region": "us-east-1"},
    )
    
    return trace


# 创建并展示一个追踪
trace = simulate_rag_agent_trace("什么是RAG技术？它的优点有哪些？")

print("=== LangSmith Trace 树形结构 ===\n")
trace.print_tree()

print(f"\n总耗时: {trace.total_duration_ms:.0f}ms")
print(f"总Token: {trace.total_tokens}")
print(f"Trace ID: {trace.trace_id}")

## 3. 反馈收集 (Feedback Collection)

In [ ]:
# 添加用户反馈
trace.add_feedback(
    key="user_rating",
    score=0.85,
    comment="回答很准确，但有点冗长",
    source="user",
)

trace.add_feedback(
    key="helpfulness",
    score=1.0,
    comment="",
    source="user",
)

trace.add_feedback(
    key="faithfulness",
    score=0.92,
    comment="自动评估: 92%声明可验证",
    source="auto_evaluator",
)

trace.add_feedback(
    key="latency_satisfaction",
    score=0.7,
    comment="响应时间偏慢 (2.3s)",
    source="system",
)

# 显示反馈
print("=== 反馈记录 ===\n")
for fb in trace.feedbacks:
    score_bar = '★' * int(fb['score'] * 5) + '☆' * (5 - int(fb['score'] * 5))
    print(f"  [{fb['source']}] {fb['key']}: {fb['score']:.2f} {score_bar}")
    if fb['comment']:
        print(f"         {fb['comment']}")

## 4. 从Trace构建Dataset

从生产追踪中自动构建评估数据集是LangSmith的关键功能。

In [ ]:
@dataclass
class LangSmithDataset:
    """模拟LangSmith Dataset"""
    dataset_id: str
    name: str
    description: str = ""
    examples: List[Dict[str, Any]] = field(default_factory=list)
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    
    def add_example(
        self,
        inputs: Dict[str, Any],
        expected_outputs: Optional[Dict[str, Any]] = None,
        metadata: Optional[Dict[str, Any]] = None,
    ) -> str:
        """添加样本"""
        example_id = str(uuid.uuid4())
        self.examples.append({
            'id': example_id,
            'inputs': inputs,
            'expected_outputs': expected_outputs or {},
            'metadata': metadata or {},
            'created_at': datetime.now().isoformat(),
        })
        return example_id
    
    def from_traces(
        self,
        traces: List[Trace],
        filter_positive_feedback: bool = True,
        min_feedback_score: float = 0.7,
    ) -> int:
        """从Trace列表自动构建数据集"""
        count = 0
        for trace in traces:
            # 过滤: 只使用高质量反馈的Trace
            if filter_positive_feedback:
                feedback_scores = [f['score'] for f in trace.feedbacks]
                if feedback_scores:
                    avg_score = sum(feedback_scores) / len(feedback_scores)
                    if avg_score < min_feedback_score:
                        continue
                elif filter_positive_feedback:
                    continue  # 没有反馈的不使用
            
            # 从Trace提取输入
            inputs = trace.root_span.inputs
            # 从Trace提取输出
            outputs = trace.root_span.outputs or {}
            
            self.add_example(
                inputs=inputs,
                expected_outputs=outputs,
                metadata={
                    'source_trace_id': trace.trace_id,
                    'user_id': trace.user_id,
                    'feedback_count': len(trace.feedbacks),
                },
            )
            count += 1
        
        return count


# 创建一些Traces并构建Dataset
traces = [
    simulate_rag_agent_trace("什么是RAG？"),
    simulate_rag_agent_trace("深度学习的应用有哪些？"),
    simulate_rag_agent_trace("Python和Java有什么区别？"),
    simulate_rag_agent_trace("如何部署机器学习模型？"),
    simulate_rag_agent_trace("什么是向量数据库？"),
]

# 添加一些反馈
for trace in traces:
    trace.add_feedback("user_rating", random.uniform(0.6, 1.0), source="user")

# 构建数据集
dataset = LangSmithDataset(
    dataset_id=str(uuid.uuid4()),
    name="RAG Production Traces v1",
    description="从生产追踪自动构建的评估数据集",
)

added = dataset.from_traces(traces, filter_positive_feedback=True, min_feedback_score=0.7)
print(f"从 {len(traces)} 个Trace中创建了 {added} 个数据集样本")
print(f"\n数据集: {dataset.name}")
print(f"样本: {len(dataset.examples)} 个")

## 5. 实验对比 (A vs B Experiment Comparison)

In [ ]:
@dataclass
class ExperimentResult:
    """"LangSmith实验对比结果"""
    experiment_name: str
    variant_a_name: str
    variant_b_name: str
    
    # 各变体的指标
    metrics_a: Dict[str, float] = field(default_factory=dict)
    metrics_b: Dict[str, float] = field(default_factory=dict)
    
    # 统计
    num_runs_a: int = 0
    num_runs_b: int = 0
    
    def compare(self) -> Dict[str, Any]:
        """对比两个变体"""
        comparison = {}
        
        all_metrics = set(self.metrics_a.keys()) | set(self.metrics_b.keys())
        for metric in sorted(all_metrics):
            val_a = self.metrics_a.get(metric, 0)
            val_b = self.metrics_b.get(metric, 0)
            delta = val_b - val_a
            pct_change = (delta / abs(val_a)) * 100 if val_a != 0 else 0
            
            comparison[metric] = {
                'a': val_a,
                'b': val_b,
                'delta': delta,
                'pct_change': round(pct_change, 1),
                'winner': 'B' if delta > 0 else 'A' if delta < 0 else 'tie',
            }
        
        return comparison
    
    def print_report(self):
        """打印对比报告"""
        print(f"\n{'='*60}")
        print(f"  实验: {self.experiment_name}")
        print(f"  {self.variant_a_name} (n={self.num_runs_a}) vs "
              f"{self.variant_b_name} (n={self.num_runs_b})")
        print(f"{'='*60}")
        print(f"\n{'指标':<25s} {'A':>8s} {'B':>8s} {'Δ':>8s} {'%':>8s} {'优胜':>6s}")
        print("-" * 65)
        
        comparison = self.compare()
        for metric, result in comparison.items():
            print(f"{metric:<25s} {result['a']:8.3f} {result['b']:8.3f} "
                  f"{result['delta']:+8.3f} {result['pct_change']:+7.1f}% "
                  f"{result['winner']:>6s}")


# 模拟A/B实验
experiment = ExperimentResult(
    experiment_name="RAG Pipeline v2 vs v3",
    variant_a_name="v2 (基线)",
    variant_b_name="v3 (混合检索)",
    num_runs_a=1000,
    num_runs_b=1000,
    metrics_a={
        'faithfulness': 0.78,
        'answer_relevancy': 0.82,
        'context_precision': 0.75,
        'context_recall': 0.70,
        'answer_completeness': 0.80,
        'avg_latency_ms': 1850,
        'avg_cost_per_query': 0.015,
    },
    metrics_b={
        'faithfulness': 0.85,
        'answer_relevancy': 0.84,
        'context_precision': 0.82,
        'context_recall': 0.78,
        'answer_completeness': 0.81,
        'avg_latency_ms': 2100,
        'avg_cost_per_query': 0.018,
    },
)

experiment.print_report()

## 6. 与评估管道集成

将LangSmith追踪与Phase 10的评估框架集成。

In [ ]:
# 导入Phase 10的评估器 (假设在同一个工作目录)
# 注意: 如果文件不存在，这会失败，但不影响演示理解

try:
    import sys
    sys.path.insert(0, '.')
    
    # 尝试导入六维度评估器
    from importlib import import_module
    # evaluator_module = import_module('01-six-dimension-evaluation')
    # MultiDimensionEvaluator = evaluator_module.MultiDimensionEvaluator
    
    print("[模拟集成] 假设已导入 MultiDimensionEvaluator")
    print("集成点:")
    print("  1. LangSmith收集Trace → 自动触发评估")
    print("  2. 评估结果作为Feedback写回LangSmith")
    print("  3. 低分Trace自动标记为需要审查")
    print("  4. 高质量Trace自动加入金标准数据集")
    
except Exception as e:
    print(f"无法导入评估器: {e}")
    print("这是在独立运行notebook时的预期行为")

## 7. 总结

LangSmith提供了完整的LLM应用可观测性:

**核心概念:**
- **Trace**: 一个完整的用户交互 → 包含多个Span
- **Span**: 原子操作 (LLM调用、检索、工具使用)
- **Feedback**: 用户或自动评估的反馈
- **Dataset**: 从生产Traces自动构建的评估数据集
- **Experiment**: A/B对比不同版本的性能

**最佳实践:**
1. 所有生产请求都应创建Trace
2. 使用有意义的Span名称和标签
3. 收集用户反馈并关联到Trace
4. 定期从高质量Trace构建评估数据集
5. 使用Experiment功能进行A/B测试

**与评估框架集成:**
```
用户请求 → Trace → 自动评估 → Feedback →
  ├─ 高质量 → 金标准数据集
  ├─ 低质量 → 人工审查
  └─ 趋势异常 → 告警
```